# einsum-contraction — ex1: predict + verify which indices get summed

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einsum-contraction`. Running the final beacon cell reports progress against the `Einsum: Index contraction semantics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einsum: Index contraction semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einsum-contraction`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einsum-contraction"
DD_SUBTOPIC = "Einsum: Index contraction semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Einsum index rules — quick refresher

`einsum` is governed by **two simple index rules**:

1. **An index that appears on BOTH sides of `->`** is preserved (carried through unchanged on every operand and the output). Think of it as a broadcasted/batch axis.
2. **An index that appears on the INPUT but NOT on the output** is *summed-contracted*: einsum multiplies aligned entries and sums them away.

**Repeated index on the same operand.** If an index appears twice on one operand (e.g. `'i i -> i'`), it pulls the **diagonal**. If `'i i ->'` with nothing on the right, it pulls the diagonal *and* sums it — that's the trace.

**Worked examples of the rules in action:**
- `'i j, j k -> i k'`: `j` repeats across operands → sum-contracted (matmul).
- `'i j -> i'`: `j` dropped from rhs → row sum.
- `'i j -> '`: all indices dropped → grand sum (scalar).
- `'i j, i j -> i j'`: nothing dropped → elementwise product (Hadamard).
- `'i, j -> i j'`: nothing repeated, both kept → outer product.

**Why this matters.** Once you internalise these two rules, you can read *any* einsum pattern from left to right and predict the output without running it. That's the whole pedagogical payoff.

### Exercise 1 — predict + verify which indices get summed

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Analyze
> LO: Analyse an einsum pattern string by listing which indices are preserved (appear on both sides) versus contracted (appear on input but not output), then verify your prediction against the actual output shape.
> Keywords: index-rules, sum-contraction, prediction
> ```

**KCs targeted:** `einsum-repeated-index-sums`, `einsum-missing-rhs-reduces`

Implement `ex1_predict_contraction(pattern, input_shapes)`.

Given:
- `pattern`: an einops einsum pattern string like `'i j, j k -> i k'`.
- `input_shapes`: a list of tuples, one per operand, giving each operand's shape.

Return a `dict` with three keys:
- `'preserved'`: sorted list of indices appearing on both sides of `->`.
- `'contracted'`: sorted list of indices appearing on input but missing from output.
- `'output_shape'`: the tuple einsum should produce, computed purely from the index rules (no torch call).

**The rules you must apply:**
- An index on input + output → preserved (size = the matching operand axis).
- An index on input only → contracted (summed away).
- All operand axes labelled with the same index must have the same size.

The test cell calls `einops.einsum` on random tensors and verifies your predicted `output_shape` matches the real result.

In [ ]:
def ex1_predict_contraction(pattern: str, input_shapes: list) -> dict:
    """Return {'preserved': [...], 'contracted': [...], 'output_shape': (...)}."""
    raise NotImplementedError()


def _test_ex1():
    # --- Case A: standard matmul ---
    out = ex1_predict_contraction('i j, j k -> i k', [(3, 4), (4, 5)])
    assert out['preserved']  == ['i', 'k'], f"preserved: {out['preserved']}"
    assert out['contracted'] == ['j'],      f"contracted: {out['contracted']}"
    assert out['output_shape'] == (3, 5),   f"shape: {out['output_shape']}"
    real = einops.einsum(t.randn(3, 4), t.randn(4, 5), 'i j, j k -> i k')
    assert tuple(real.shape) == out['output_shape']

    # --- Case B: row sum (j dropped from rhs) ---
    out = ex1_predict_contraction('i j -> i', [(2, 7)])
    assert out['preserved']  == ['i']
    assert out['contracted'] == ['j']
    assert out['output_shape'] == (2,)
    real = einops.einsum(t.randn(2, 7), 'i j -> i')
    assert tuple(real.shape) == out['output_shape']

    # --- Case C: batched matmul (b preserved through both) ---
    out = ex1_predict_contraction('b i k, b k j -> b i j', [(2, 3, 4), (2, 4, 5)])
    assert out['preserved']  == ['b', 'i', 'j']
    assert out['contracted'] == ['k']
    assert out['output_shape'] == (2, 3, 5)
    real = einops.einsum(t.randn(2, 3, 4), t.randn(2, 4, 5), 'b i k, b k j -> b i j')
    assert tuple(real.shape) == out['output_shape']

    # --- Case D: outer product (nothing repeated, both kept) ---
    out = ex1_predict_contraction('i, j -> i j', [(3,), (5,)])
    assert out['preserved']  == ['i', 'j']
    assert out['contracted'] == []
    assert out['output_shape'] == (3, 5)
    real = einops.einsum(t.randn(3), t.randn(5), 'i, j -> i j')
    assert tuple(real.shape) == out['output_shape']

    # --- Case E: hadamard (nothing dropped → no contraction) ---
    out = ex1_predict_contraction('i j, i j -> i j', [(4, 6), (4, 6)])
    assert out['contracted'] == []
    assert out['output_shape'] == (4, 6)

    print('all 5 cases predicted correctly')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_predict_contraction(pattern: str, input_shapes: list) -> dict:
    lhs, rhs = pattern.split('->')
    rhs_idx = rhs.strip().split()
    operand_idx = [op.strip().split() for op in lhs.split(',')]
    # All input indices (across operands), deduped, sorted.
    all_input = set()
    for ops in operand_idx:
        all_input.update(ops)
    preserved = sorted(i for i in all_input if i in rhs_idx)
    contracted = sorted(i for i in all_input if i not in rhs_idx)
    # Map each named index to its size (look up in the first operand that has it).
    sizes = {}
    for ops, shape in zip(operand_idx, input_shapes):
        for name, n in zip(ops, shape):
            sizes.setdefault(name, n)
    output_shape = tuple(sizes[i] for i in rhs_idx)
    return {'preserved': preserved, 'contracted': contracted, 'output_shape': output_shape}
```

**Why this is an Analyse-level task.** You're not running einsum and reading the shape off — you're *predicting* the shape from the pattern string alone. That forces you to internalise the two rules: shared-with-rhs survives, missing-from-rhs gets summed.

**The size-map trick.** When the same name appears on multiple operands (`'i j, j k -> i k'`), all occurrences must have the same size — einsum will error otherwise. `sizes.setdefault` records the first occurrence; you could optionally assert that later occurrences match for an even stricter checker.

**What about repeated index on one operand?** `'i i -> i'` is the diagonal trick; einops handles it but this simple predictor doesn't. The standard ARENA patterns never use that form, so it's out of scope here.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()